# Data Preprocessing (Stage 2) — Split, Imputation, Duplicates, Encoding

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv('../data/processed/cars_stage1_cleaned.csv')

print(df.shape)
df.head()

(6016, 13)


,Location,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price,Mileage_Unit,Car_Age,Brand
0,Mumbai,72000,CNG,Manual,First,26.60,998.0,58.16,5.0,1.75,km/kg,10,Maruti
1,Pune,41000,Diesel,Manual,First,19.67,1582.0,126.20,5.0,12.50,kmpl,5,Hyundai
2,Chennai,46000,Petrol,Manual,First,18.20,1199.0,88.70,5.0,4.50,kmpl,9,Honda
3,Chennai,87000,Diesel,Manual,First,20.77,1248.0,88.76,7.0,6.00,kmpl,8,Maruti
4,Coimbatore,40670,Diesel,Automatic,Second,15.20,1968.0,140.80,5.0,17.74,kmpl,7,Audi


# Spliting

In [3]:
X = df.drop("Price", axis=1)
y = df["Price"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

checking duplicates before imputation

In [5]:
X_train.assign(Price=y_train).duplicated().sum()

np.int64(2)

# Imputation

##### we will use group-based imputation to customize the imputation values

##### like if we impute with mean in Power, there is too much variety between Brands' Power

In [6]:
X_train.isnull().sum()

Location               0
Kilometers_Driven      0
Fuel_Type              0
Transmission           0
Owner_Type             0
Mileage                0
Engine                30
Power                112
Seats                 35
Mileage_Unit           0
Car_Age                0
Brand                  0
dtype: int64

In [7]:
X_test.isnull().sum()

Location              0
Kilometers_Driven     0
Fuel_Type             0
Transmission          0
Owner_Type            0
Mileage               0
Engine                6
Power                31
Seats                 8
Mileage_Unit          0
Car_Age               0
Brand                 0
dtype: int64

##### Fuel_Type determines Mileage_Unit with 100% consistency (CNG/LPG -> km/kg, Diesel/Petrol -> kmpl)
##### Electric cars are absent here because they were already dropped in Stage 1 (their Mileage/Mileage_Unit had no meaningful value)

## Mileage_Unit imputation based on Fuel_Type

In [8]:
X_train.head()

,Location,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Mileage_Unit,Car_Age,Brand
4675,Pune,49700,Diesel,Manual,Second,21.90,1396.0,88.76,5.0,kmpl,8,Hyundai
1204,Hyderabad,96000,Diesel,Automatic,First,12.35,2179.0,187.74,5.0,kmpl,8,Land Rover
5640,Pune,125000,Petrol,Manual,Second,18.90,998.0,67.10,5.0,kmpl,17,Maruti
4638,Delhi,75348,Diesel,Manual,First,25.80,1498.0,98.60,5.0,kmpl,7,Honda
5562,Bangalore,97000,Diesel,Automatic,Second,19.08,1582.0,126.32,5.0,kmpl,8,Hyundai


In [9]:
print(X_train['Fuel_Type'].value_counts())

Fuel_Type
Diesel    2580
Petrol    2180
CNG         42
LPG         10
Name: count, dtype: int64


In [10]:
pd.crosstab(X_train['Fuel_Type'], X_train['Mileage_Unit'])

Mileage_Unit,km/kg,kmpl
Fuel_Type,,
CNG,42,0
Diesel,0,2580
LPG,10,0
Petrol,0,2180


In [11]:
print(X_train['Mileage_Unit'].isnull().sum())
print(X_test['Mileage_Unit'].isnull().sum())

0
0


In [12]:
print(X_train['Mileage'].isnull().sum())
print(X_test['Mileage'].isnull().sum())

0
0


##### Not needed here (Electric rows already dropped, so Mileage_Unit has 0 nulls).
##### Kept as reference: if Mileage_Unit had nulls, we'd fill it using this fixed
##### real-world mapping (Fuel_Type -> Unit), not a learned statistic — so it's
##### safe to apply on X_train and X_test separately.
fuel_to_unit = {'CNG': 'km/kg', 'LPG': 'km/kg', 'Diesel': 'kmpl', 'Petrol': 'kmpl'}
X_train['Mileage_Unit'] = X_train['Mileage_Unit'].fillna(X_train['Fuel_Type'].map(fuel_to_unit))
X_test['Mileage_Unit'] = X_test['Mileage_Unit'].fillna(X_test['Fuel_Type'].map(fuel_to_unit))

##### Mileage/Mileage_Unit already have 0 nulls here, the rows that used to be null (Electric cars) were removed in Stage 1, before the split

## Engine imputation by median based on Brand

In [13]:
brand_median_engine = X_train.groupby('Brand')['Engine'].median() # customized fillna

overall_median_engine = X_train['Engine'].median() # filling remaining NaNs with overall median(if a brand has no value to calc median)

In [14]:
X_train['Engine'] = X_train['Engine'].fillna(X_train['Brand'].map(brand_median_engine))
X_train['Engine'] = X_train['Engine'].fillna(overall_median_engine)

# transfrom only
X_test['Engine'] = X_test['Engine'].fillna(X_test['Brand'].map(brand_median_engine))
X_test['Engine'] = X_test['Engine'].fillna(overall_median_engine)

In [15]:
print(X_train['Engine'].isnull().sum())
print(X_test['Engine'].isnull().sum())

0
0


## Power imputation by median based on Brand

In [16]:
# same as Engine:

brand_median_power = X_train.groupby('Brand')['Power'].median() # customized fillna

overall_median_power = X_train['Power'].median() # filling remaining NaNs with overall median (if a brand has no value to calc median)

In [17]:
X_train['Power'] = X_train['Power'].fillna(X_train['Brand'].map(brand_median_power))
X_train['Power'] = X_train['Power'].fillna(overall_median_power)

# transform only
X_test['Power'] = X_test['Power'].fillna(X_test['Brand'].map(brand_median_power))
X_test['Power'] = X_test['Power'].fillna(overall_median_power)

In [18]:
print(X_train['Power'].isnull().sum())
print(X_test['Power'].isnull().sum())

0
0


## Seats imputation using mode based on Brand same as Power and Engine

In [19]:
# Seats: same pattern as Power, but using mode (categorical-like), with empty-mode check

brand_mode_seats = X_train.groupby('Brand')['Seats'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else np.nan
)  # customized fillna

overall_mode_seats = X_train['Seats'].mode()[0]  # filling remaining NaNs with overall mode (if a brand has no value to calc mode)

In [20]:
X_train['Seats'] = X_train['Seats'].fillna(X_train['Brand'].map(brand_mode_seats))
X_train['Seats'] = X_train['Seats'].fillna(overall_mode_seats)

# map only
X_test['Seats'] = X_test['Seats'].fillna(X_test['Brand'].map(brand_mode_seats))
X_test['Seats'] = X_test['Seats'].fillna(overall_mode_seats)

In [21]:
print(X_train['Seats'].isnull().sum())
print(X_test['Seats'].isnull().sum())

0
0


### **well done**

# Duplicates

In [22]:
print(X_train.duplicated().sum())
print(X_test.duplicated().sum())

15
1


are they duplicated in x only or x,y????

In [23]:
X_train.assign(Price=y_train).duplicated().sum()

np.int64(2)

In [24]:
combined_test = X_test.assign(Price=y_test)
print(combined_test.duplicated().sum())

0


there are only 2 real dupliacates

now we drop the fully duplicated rows

In [25]:
# Combine X and y temporarily to find rows that are fully duplicated (X and Price together)
combined = X_train.assign(Price=y_train)

# Find which rows are duplicates (keep='first' keeps the first occurrence, marks the rest as True)
is_duplicate = combined.duplicated()

# Keep only rows that are NOT duplicates
X_train = X_train[~is_duplicate]
y_train = y_train[~is_duplicate]

In [26]:
X_train.assign(Price=y_train).duplicated().sum()

np.int64(0)

#### duplicated before cleaning was 0, after is 2, thats means that there were 4 rows were the same except in an one feature or more, but when we cleaned the data or dropped columns, they become perfectly duplicated.

# Encoding

### Encoding plan:
##### Transmission -> One-Hot (drop_first), 2 values, no order
##### Mileage_Unit -> One-Hot (drop_first), 2 values, no order
##### Owner_Type -> Ordinal Encoding, First < Second < Third < Fourth
##### Fuel_Type -> One-Hot, no natural order
##### Location -> One-Hot, 11 values, no order
##### Brand -> One-Hot, grouped rare brands into 'Other' first (15 categories)
##### Seats -> no encoding, kept as numeric (has real magnitude, e.g. 7 seats > 5 seats)

In [27]:
X_train['Brand'].value_counts()

Brand
Maruti           971
Hyundai          884
Honda            480
Toyota           332
Mercedes-Benz    256
Volkswagen       251
Ford             241
Mahindra         226
BMW              205
Audi             191
Tata             147
Skoda            139
Renault          117
Chevrolet         90
Nissan            75
Land Rover        51
Jaguar            31
Mini              21
Mitsubishi        21
Fiat              20
Volvo             18
Porsche           13
Jeep              12
Datsun            11
Isuzu              3
Force              2
Bentley            1
Lamborghini        1
Name: count, dtype: int64

### Brand encoding
Rare brands (under 100 occurrences) are grouped into 'Other'. The threshold and the list of rare brands are computed from **X_train only**, then the *same* list is applied to X_test (no new computation on X_test).

In [28]:
brand_counts = X_train['Brand'].value_counts()  # computed from X_train only
rare_brands = brand_counts[brand_counts < 100].index

X_train['Brand'] = X_train['Brand'].replace(rare_brands, 'Other')
X_test['Brand'] = X_test['Brand'].replace(rare_brands, 'Other')  # apply same list, no new computation

In [29]:
print(sorted(X_train['Location'].unique()))
print(sorted(X_train['Brand'].unique()))

['Ahmedabad', 'Bangalore', 'Chennai', 'Coimbatore', 'Delhi', 'Hyderabad', 'Jaipur', 'Kochi', 'Kolkata', 'Mumbai', 'Pune']
['Audi', 'BMW', 'Ford', 'Honda', 'Hyundai', 'Mahindra', 'Maruti', 'Mercedes-Benz', 'Other', 'Renault', 'Skoda', 'Tata', 'Toyota', 'Volkswagen']


In [30]:
dropdown_options = {
    'Location': ['Ahmedabad', 'Bangalore', 'Chennai', 'Coimbatore', 'Delhi', 'Hyderabad', 'Jaipur', 'Kochi', 'Kolkata', 'Mumbai', 'Pune'],
    'Fuel_Type': ['CNG', 'Diesel', 'LPG', 'Petrol'],
    'Transmission': ['Automatic', 'Manual'],
    'Owner_Type': ['First', 'Second', 'Third', 'Fourth & Above'],
    'Brand': ['Audi', 'BMW', 'Ford', 'Honda', 'Hyundai', 'Mahindra', 'Maruti', 'Mercedes-Benz', 'Renault', 'Skoda', 'Tata', 'Toyota', 'Volkswagen']
}

In [31]:
print(X_train['Brand'].nunique())
print(X_train['Brand'].value_counts())

14
Brand
Maruti           971
Hyundai          884
Honda            480
Other            370
Toyota           332
Mercedes-Benz    256
Volkswagen       251
Ford             241
Mahindra         226
BMW              205
Audi             191
Tata             147
Skoda            139
Renault          117
Name: count, dtype: int64


#### One-Hot Encoding (Transmission, Mileage_Unit, Fuel_Type, Location, Brand)
We one-hot encode X_train and X_test **separately**, then align X_test's columns to match X_train exactly. This guarantees:
- if a category exists only in X_test (never seen in X_train), it's dropped (the model never learned a weight for it anyway)
- if a category exists only in X_train, X_test gets a 0 column for it

No statistic from X_test is used to decide X_train's columns — only X_train's own categories define the final column set.

In [32]:
onehot_cols = ['Transmission', 'Mileage_Unit', 'Fuel_Type', 'Location', 'Brand']

X_train = pd.get_dummies(X_train, columns=onehot_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=onehot_cols, drop_first=True)

# Align X_test's columns to X_train's columns (X_train defines the reference set)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(X_train.shape)
print(X_test.shape)

(4810, 35)
(1204, 35)


In [33]:
X_train.head()

,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,Car_Age,Transmission_Manual,Mileage_Unit_kmpl,Fuel_Type_Diesel,...,Brand_Hyundai,Brand_Mahindra,Brand_Maruti,Brand_Mercedes-Benz,Brand_Other,Brand_Renault,Brand_Skoda,Brand_Tata,Brand_Toyota,Brand_Volkswagen
4675,49700,Second,21.90,1396.0,88.76,5.0,8,True,True,True,...,True,False,False,False,False,False,False,False,False,False
1204,96000,First,12.35,2179.0,187.74,5.0,8,False,True,True,...,False,False,False,False,True,False,False,False,False,False
5640,125000,Second,18.90,998.0,67.10,5.0,17,True,True,False,...,False,False,True,False,False,False,False,False,False,False
4638,75348,First,25.80,1498.0,98.60,5.0,7,True,True,True,...,False,False,False,False,False,False,False,False,False,False
5562,97000,Second,19.08,1582.0,126.32,5.0,8,False,True,True,...,True,False,False,False,False,False,False,False,False,False


In [34]:
print(X_train.columns.tolist())

['Kilometers_Driven', 'Owner_Type', 'Mileage', 'Engine', 'Power', 'Seats', 'Car_Age', 'Transmission_Manual', 'Mileage_Unit_kmpl', 'Fuel_Type_Diesel', 'Fuel_Type_LPG', 'Fuel_Type_Petrol', 'Location_Bangalore', 'Location_Chennai', 'Location_Coimbatore', 'Location_Delhi', 'Location_Hyderabad', 'Location_Jaipur', 'Location_Kochi', 'Location_Kolkata', 'Location_Mumbai', 'Location_Pune', 'Brand_BMW', 'Brand_Ford', 'Brand_Honda', 'Brand_Hyundai', 'Brand_Mahindra', 'Brand_Maruti', 'Brand_Mercedes-Benz', 'Brand_Other', 'Brand_Renault', 'Brand_Skoda', 'Brand_Tata', 'Brand_Toyota', 'Brand_Volkswagen']


##### converting True, False into 1,0

In [35]:
bool_cols = X_train.select_dtypes(include='bool').columns
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

In [36]:
X_train.head()

,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,Car_Age,Transmission_Manual,Mileage_Unit_kmpl,Fuel_Type_Diesel,...,Brand_Hyundai,Brand_Mahindra,Brand_Maruti,Brand_Mercedes-Benz,Brand_Other,Brand_Renault,Brand_Skoda,Brand_Tata,Brand_Toyota,Brand_Volkswagen
4675,49700,Second,21.90,1396.0,88.76,5.0,8,1,1,1,...,1,0,0,0,0,0,0,0,0,0
1204,96000,First,12.35,2179.0,187.74,5.0,8,0,1,1,...,0,0,0,0,1,0,0,0,0,0
5640,125000,Second,18.90,998.0,67.10,5.0,17,1,1,0,...,0,0,1,0,0,0,0,0,0,0
4638,75348,First,25.80,1498.0,98.60,5.0,7,1,1,1,...,0,0,0,0,0,0,0,0,0,0
5562,97000,Second,19.08,1582.0,126.32,5.0,8,0,1,1,...,1,0,0,0,0,0,0,0,0,0


### Ordinal Encoding for Owner_Type
This is a fixed real-world mapping (not a learned statistic), so it's safe to apply directly on X_train and X_test separately — same reasoning as the Fuel_Type -> Mileage_Unit mapping earlier.

In [37]:
X_train['Owner_Type'].unique()

<ArrowStringArray>
['Second', 'First', 'Third', 'Fourth & Above']
Length: 4, dtype: str

In [38]:
owner_order = {'First': 1, 'Second': 2, 'Third': 3, 'Fourth & Above': 4}
X_train['Owner_Type'] = X_train['Owner_Type'].map(owner_order)
X_test['Owner_Type'] = X_test['Owner_Type'].map(owner_order)

In [39]:
print(X_train['Owner_Type'].unique())
print(X_test['Owner_Type'].unique())

[2 1 3 4]
[2 1 3 4]


## MinMaxScaler

In [40]:
numeric_cols = ['Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Car_Age']

scaler = MinMaxScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])   # fit + transform 
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])          # transform

# Save final processed data
Saved separately (not merged back into one file) so the train/test split is preserved for modeling notebooks.

In [41]:
X_train.to_csv('../data/processed/X_train_processed.csv', index=False)
X_test.to_csv('../data/processed/X_test_processed.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(4810, 35) (1204, 35) (4810,) (1204,)


In [42]:
X_train.head()

,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,Car_Age,Transmission_Manual,Mileage_Unit_kmpl,Fuel_Type_Diesel,...,Brand_Hyundai,Brand_Mahindra,Brand_Maruti,Brand_Mercedes-Benz,Brand_Other,Brand_Renault,Brand_Skoda,Brand_Tata,Brand_Toyota,Brand_Volkswagen
4675,0.063922,2,0.652952,0.143655,0.103766,5.0,0.333333,1,1,1,...,1,0,0,0,0,0,0,0,0,0
1204,0.123678,1,0.368217,0.289356,0.292012,5.0,0.333333,0,1,1,...,0,0,0,0,1,0,0,0,0,0
5640,0.161105,2,0.563506,0.069594,0.062571,5.0,0.761905,1,1,0,...,0,0,1,0,0,0,0,0,0,0
4638,0.097024,1,0.769231,0.162635,0.122480,5.0,0.285714,1,1,1,...,0,0,0,0,0,0,0,0,0,0
5562,0.124968,2,0.568873,0.178266,0.175200,5.0,0.333333,0,1,1,...,1,0,0,0,0,0,0,0,0,0


## saving for the model

In [43]:
import joblib

joblib.dump(rare_brands, '../models/rare_brands.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(X_train.columns.tolist(), '../models/feature_columns.pkl')
joblib.dump(dropdown_options, '../models/dropdown_options.pkl')

['../models/dropdown_options.pkl']

# Data Preprocessing Summary

- Dropped `S.No.` (useless index) and `New_Price` (86% missing)
- Dropped rows with missing `Price` (target — can't impute it)
- Split unit text from numbers in `Mileage`, `Engine`, `Power` (handled a hidden "null bhp" value too)
- Dropped Electric cars (no meaningful Mileage) — done before the split
- Engineered `Car_Age` (from Year) and `Brand` (from Name)
- Split into train/test, then did everything below using X_train only (no leakage):
  - Imputed `Engine`/`Power` with median per Brand, `Seats` with mode per Brand (fallback: overall median/mode)
  - Removed true duplicate rows (X and Price both identical)
  - Grouped rare brands (<100 occurrences) into "Other"
  - One-hot encoded categorical columns, ordinal encoded `Owner_Type`
  - Scaled numeric columns (`Kilometers_Driven`, `Mileage`, `Engine`, `Power`, `Car_Age`) with MinMaxScaler

Final: train/test sets, fully numeric, no missing values, scaled and ready for modeling.